In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [1]:
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import ByteLevel
from tokenizers.decoders import ByteLevel as ByteLevelDecoder

tokenizer = Tokenizer(BPE(unk_token="<unk>"))
tokenizer.pre_tokenizer = ByteLevel(add_prefix_space=False)
tokenizer.decoder = ByteLevelDecoder()

trainer = BpeTrainer(
    vocab_size=50304,
    special_tokens=["<unk>"],
)

tokenizer.train(
    files=["llama2_uncensored_finetuning_dataset.txt"],
    trainer=trainer,
)

tokenizer.save("tokenizer.json")

print("Complete")




Complete


In [3]:
class Net(nn.Module):
    def __init__(self, input_features: int):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(input_features, input_features),
            nn.GELU(approximate = "tanh"),
            nn.Linear(input_features, input_features),
        )

    def forward(self, x):
        return self.mlp(x)

In [4]:
class Attention(nn.Module):
    def __init__(self, dim: int, heads: int = 8):
        super().__init__()
        self.heads = heads
        self.dim = dim
        self.head_dim = dim // heads
        
        self.key = nn.Linear(dim, dim)
        self.query = nn.Linear(dim, dim)
        self.value = nn.Linear(dim, dim)
        self.proj = nn.Linear(dim, dim)

    def forward(self, x):
        B, T, C = x.shape

        mask = torch.tril(
            torch.ones(T, T, device = x.device, dtype = torch.bool)
        )

        q = (
            self.query(x)
            .reshape(B, T, self.heads, self.head_dim)
            .permute(0, 2, 1, 3)
        )

        k = (
            self.key(x)
            .reshape(B, T, self.heads, self.head_dim)
            .permute(0, 2, 1, 3)
        )

        v = (
            self.value(x)
            .reshape(B, T, self.heads, self.head_dim)
            .permute(0, 2, 1, 3)
        )

        q = q * (self.head_dim ** -0.5)
        attention = q @ k.transpose(-2, -1)
        attention = attention.masked_fill(~mask, float("-inf"))
        attention = attention.softmax(dim = -1)
        x = attention @ v

        x = x.transpose(1, 2).reshape(B, T, C)
        x = self.proj(x)
        return x
        

In [5]:
class TransformerBlock(nn.Module):
    def __init__(self, hidden_size: int, heads: int):
        super().__init__()
        self.norm1 = nn.LayerNorm(hidden_size)
        self.attention = Attention(hidden_size, heads = heads)
        self.norm2 = nn.LayerNorm(hidden_size)
        self.mlp = Net(hidden_size)

    def forward(self, x):
        x = x + self.attention(self.norm1(x))
        x = x + self.mlp(self.norm2(x))

        return x


class NBlocks(nn.Module):
    def __init__(self, num_blocks: int, hidden_size: int, heads: int, vocab_size: int):
        super().__init__()
        self.transformer_blocks = nn.Sequential(
            *[TransformerBlock(hidden_size, heads) for _ in range(num_blocks)],
        )

        self.token_embedding = nn.Embedding(vocab_size, hidden_size)
        self.positional_embedding = nn.Embedding(block_size, hidden_size)

        self.final_norm = nn.LayerNorm(hidden_size)
        self.output_layer = nn.Linear(hidden_size, vocab_size)

    def forward(self, x):
        B, T = x.shape

        positions = torch.arange(T, device = x.device)

        x = self.token_embedding(x) + self.positional_embedding(positions)

        x = self.transformer_blocks(x)
        x = self.final_norm(x)
        return self.output_layer(x)

In [6]:
def generate(
    model: nn.Module,
    idx: torch.Tensor,
    max_new_tokens: int,
    block_size: int,
) -> torch.Tensor:
    model.eval()

    device = next(model.parameters()).device
    idx = idx.to(device)

    with torch.no_grad():
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]

            logits = model(idx_cond)

            logits = logits[:, -1, :]

            probs = F.softmax(logits, dim=-1)

            idx_next = torch.multinomial(
                probs,
                num_samples=1,
            )

            idx = torch.cat((idx, idx_next), dim=1)

    return idx

In [7]:
def load_dataset(path: str):
    with open(path, "r", encoding = "utf-8") as f:
        return f.read()

In [8]:
data = load_dataset("llama2_uncensored_finetuning_dataset.txt")
chars = sorted(set(data))


from tokenizers import Tokenizer

tokenizer = Tokenizer.from_file("tokenizer.json")

encode = lambda text: tokenizer.encode(text).ids
decode = lambda ids: tokenizer.decode(ids)

vocab_size = tokenizer.get_vocab_size()

data = torch.tensor(encode(data))

block_size = 256
batch_size = 32
steps = 10000
learning_rate = 3e-4

split = int(0.9 * len(data))

training_set = data[:split]
testing_set = data[split:]

def select_random(block_size, batch_size, array: torch.Tensor, device):
    lower_bounds = torch.randint(0, len(array) - block_size, (batch_size,))

    x = torch.stack([array[i: i + block_size] for i in lower_bounds]).to(device)
    y = torch.stack([array[i + 1: i + block_size + 1] for i in lower_bounds]).to(device)

    return x, y

print("Got past")

Got past


In [ ]:
device = torch.device("mps")

torch.manual_seed(42)

num_heads = 6
num_blocks = 12
dim = 384

model = NBlocks(num_blocks, dim, num_heads, vocab_size).to(device)
optim = torch.optim.AdamW(model.parameters(), lr = learning_rate)
lossf = nn.CrossEntropyLoss()

save_freq = 5000
log_freq = 300

lowest_loss = 5000

model.train()
for i in range(steps):
    random_selected = select_random(block_size, batch_size, training_set, device)
    random_selected = torch.stack(random_selected)

    optim.zero_grad()
    pred = model(random_selected[0])
    loss = lossf(pred.transpose(1, 2), random_selected[1])
    loss.backward()

    nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

    optim.step()

    if (i + 1) % log_freq == 0:
        print(f"Step {i + 1}: loss = {loss.item():.4f}")
        if loss.item() < lowest_loss:
            lowest_loss = loss.item()
            torch.save(model.state_dict(), "model_weights.pth")

    if (i + 1) % save_freq == 0:
        torch.save(model.state_dict(), "model_weights.pth")

print(f"Lowest loss: {lowest_loss}")

In [ ]:
state_dict = torch.load('model_weights.pth', weights_only = False, map_location = device)
eval_model = NBlocks(num_blocks = num_blocks, hidden_size = dim, heads = num_heads, vocab_size = vocab_size).to(device)
eval_model.load_state_dict(state_dict)
eval_model.eval()

prompt = "Hello"

total_params = sum(p.numel() for p in eval_model.parameters())
print(f"Total parameters: {total_params:,}")

start = torch.tensor(
    [encode(prompt)],
    dtype=torch.long,
)

generated = generate(
    eval_model,
    start,
    max_new_tokens=500,
    block_size=block_size,
)

print(decode(generated[0].tolist()))